# Classical Logical Reasoning System
AMAN KUMAR | M26MAC001

Input facts and rules are converted to logical clauses. Forward chaining,
backward chaining, resolution and DPLL determine whether the query follows.
Run all cells in order. Keep the seven dataset files in the Data folder.


## 1. Representation and formal input
Terms, literals and formula trees represent the supplied logic.


In [1]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from itertools import count, product
import heapq
import re
import time


class InputError(ValueError):
    """The input is outside the documented grammar."""


class SearchLimit(RuntimeError):
    """Search stopped without proving either outcome."""


@dataclass(frozen=True, order=True)
class Term:
    name: str
    args: tuple = ()
    variable: bool = False

    def __str__(self):
        if self.variable:
            return '?' + self.name
        return self.name + ('(' + ','.join(map(str, self.args)) + ')' if self.args else '')


@dataclass(frozen=True, order=True)
class Literal:
    predicate: str
    args: tuple = ()
    negative: bool = False

    def opposite(self):
        return Literal(self.predicate, self.args, not self.negative)

    def __str__(self):
        atom = self.predicate + ('(' + ','.join(map(str, self.args)) + ')' if self.args else '')
        return ('~' if self.negative else '') + atom


def atom(name, *args):
    return ('atom', Literal(name, tuple(args)))


def term_variables(term):
    return {term} if term.variable else set().union(*(term_variables(a) for a in term.args))


def free_variables(formula):
    op = formula[0]
    if op == 'atom':
        return set().union(*(term_variables(t) for t in formula[1].args))
    if op in ('all', 'exists'):
        return free_variables(formula[2]) - {formula[1]}
    return set().union(*(free_variables(f) for f in formula[1:]))


def substitute_term(term, substitution):
    if term.variable and term in substitution:
        return substitute_term(substitution[term], substitution)
    return Term(term.name, tuple(substitute_term(a, substitution) for a in term.args), term.variable)


def substitute_literal(literal, substitution):
    return Literal(literal.predicate, tuple(substitute_term(a, substitution) for a in literal.args), literal.negative)


IDENTIFIER = r'\??\w+(?:[.\x27’\-]\w+)*'
TOKEN = re.compile(r'\s*(<=>|<->|=>|->|!=|&&|\|\||[∀∃¬∧∨→↔⟷⊕≠~!&|^=(),.:]|' + IDENTIFIER + ')', re.UNICODE)
ALIASES = {'∀': 'all', 'forall': 'all', '∃': 'exists', '¬': '~', 'not': '~', '!': '~',
           '∧': '&', 'and': '&', '&&': '&', '∨': '|', 'or': '|', '||': '|',
           '→': '->', '=>': '->', 'implies': '->', '↔': '<->', '⟷': '<->',
           '<=>': '<->', 'iff': '<->', '⊕': '^', 'xor': '^', '≠': '!='}


class FormulaParser:
    """Recursive descent with precedence: ~, &, |, xor, ->, <->.

    A quantifier binds the entire following formula. Write explicit parentheses
    when combining a quantified formula with another formula.
    """
    def __init__(self, text):
        text = text.strip().rstrip('.')
        self.tokens, position = [], 0
        while position < len(text):
            match = TOKEN.match(text, position)
            if not match:
                raise InputError('Unexpected syntax near: ' + text[position:position + 45])
            token = match.group(1)
            self.tokens.append(ALIASES.get(token, token))
            position = match.end()
        self.index = 0
        self.bound = set()

    def peek(self):
        return self.tokens[self.index] if self.index < len(self.tokens) else None

    def pop(self, expected=None):
        token = self.peek()
        if token is None or (expected is not None and token != expected):
            raise InputError(f'Expected {expected or "token"}, found {token}')
        self.index += 1
        return token

    def parse(self):
        result = self.expression()
        if self.peek() is not None:
            raise InputError('Extra tokens: ' + ' '.join(self.tokens[self.index:]))
        # Explicit ?variables and conventional x/y/z are universally closed.
        for variable in sorted(free_variables(result), reverse=True):
            result = ('all', variable, result)
        return result

    def expression(self, minimum=0):
        left = self.unary()
        precedence = {'<->': 1, '->': 2, '^': 3, '|': 4, '&': 5}
        names = {'<->': 'iff', '->': 'implies', '^': 'xor', '|': 'or', '&': 'and'}
        while self.peek() in precedence and precedence[self.peek()] >= minimum:
            operator = self.pop()
            level = precedence[operator]
            right = self.expression(level if operator == '->' else level + 1)
            left = (names[operator], left, right)
        return left

    def unary(self):
        token = self.peek()
        if token == '~':
            self.pop()
            return ('not', self.unary())
        if token in ('all', 'exists'):
            quantifier = self.pop()
            variables = [self.pop()]
            while self.peek() == ',':
                self.pop()
                variables.append(self.pop())
            for name in variables:
                if not re.fullmatch(r'\??[^\W\d]\w*', name):
                    raise InputError('Invalid bound variable: ' + name)
            if self.peek() in ('.', ':'):
                self.pop()
            old_bound = self.bound.copy()
            self.bound.update(v.lstrip('?') for v in variables)
            body = self.expression()
            self.bound = old_bound
            for name in reversed(variables):
                body = (quantifier, Term(name.lstrip('?'), variable=True), body)
            return body
        if token == '(':
            self.pop()
            formula = self.expression()
            self.pop(')')
            return formula
        left = self.term()
        if self.peek() in ('=', '!='):
            operator = self.pop()
            result = atom('=', left, self.term())
            return ('not', result) if operator == '!=' else result
        return atom(left.name, *left.args)

    def term(self):
        name = self.pop()
        if not re.fullmatch(IDENTIFIER, name) or name in ('all', 'exists'):
            raise InputError('Invalid symbol: ' + name)
        if name.startswith('__'):
            raise InputError('Names beginning __ are reserved for the reasoner')
        arguments = []
        if self.peek() == '(':
            self.pop()
            if self.peek() != ')':
                arguments.append(self.term())
                while self.peek() == ',':
                    self.pop()
                    arguments.append(self.term())
            self.pop(')')
        variable = not arguments and (name.startswith('?') or name in self.bound or name in {'x', 'y', 'z', 'u', 'v', 'w', 'X', 'Y', 'Z'})
        return Term(name.lstrip('?'), tuple(arguments), variable)


def parse_formula(text):
    return FormulaParser(text).parse()


## 2. Prepare clauses
CNF conversion and grounding prepare formulas for inference.


In [2]:
from __future__ import annotations

def negation_normal_form(formula, negate=False):
    """Remove implications and move negations down to atoms."""
    op = formula[0]
    if op == 'atom':
        literal = formula[1]
        return ('atom', literal.opposite() if negate else literal)
    if op == 'not':
        return negation_normal_form(formula[1], not negate)
    if op == 'implies':
        return negation_normal_form(('or', ('not', formula[1]), formula[2]), negate)
    if op in ('iff', 'xor'):
        a, b = formula[1:]
        equivalent = ('and', ('or', ('not', a), b), ('or', ('not', b), a))
        return negation_normal_form(equivalent, negate ^ (op == 'xor'))
    if op in ('all', 'exists'):
        flipped = {'all': 'exists', 'exists': 'all'}[op] if negate else op
        return (flipped, formula[1], negation_normal_form(formula[2], negate))
    operator = {'and': 'or', 'or': 'and'}[op] if negate else op
    return (operator, negation_normal_form(formula[1], negate), negation_normal_form(formula[2], negate))


def clause_variables(clause):
    return set().union(*(term_variables(t) for literal in clause for t in literal.args))


def tautology(clause):
    # Only EXACT complements are tautological; unifiable ones need not be.
    return any(literal.opposite() in clause for literal in clause)


class Clausifier:
    def __init__(self, max_clauses=12000):
        self.serial = count()
        self.max_clauses = max_clauses

    def skolemize(self, formula, universals=(), environment=None):
        environment = {} if environment is None else environment
        op = formula[0]
        if op == 'atom':
            return ('atom', substitute_literal(formula[1], environment))
        if op in ('all', 'exists'):
            variable = formula[1]
            if op == 'all':
                fresh = Term('__v' + str(next(self.serial)), variable=True)
                return self.skolemize(formula[2], universals + (fresh,), {**environment, variable: fresh})
            # Every universal in scope is a possible dependency of the witness.
            witness = Term('__sk' + str(next(self.serial)), universals)
            return self.skolemize(formula[2], universals, {**environment, variable: witness})
        return (op, *(self.skolemize(f, universals, environment) for f in formula[1:]))

    def cnf(self, formula):
        if formula[0] == 'atom':
            return [frozenset([formula[1]])]
        left, right = self.cnf(formula[1]), self.cnf(formula[2])
        if formula[0] == 'and':
            clauses = left + right
        else:
            if len(left) * len(right) > self.max_clauses:
                raise SearchLimit('CNF distribution exceeded clause budget')
            clauses = [a | b for a in left for b in right if not tautology(a | b)]
        if len(clauses) > self.max_clauses:
            raise SearchLimit('CNF exceeded clause budget')
        return list(dict.fromkeys(clauses))

    def convert(self, formulas):
        clauses = []
        for formula in formulas:
            clauses.extend(self.cnf(self.skolemize(negation_normal_form(formula))))
        return list(dict.fromkeys(c for c in clauses if not tautology(c)))


@dataclass
class Budget:
    seconds: float = 2.0
    max_ground_clauses: int = 50000
    max_resolution_clauses: int = 4000
    deadline: float = field(init=False)

    def __post_init__(self):
        self.deadline = time.perf_counter() + self.seconds

    def check(self):
        if time.perf_counter() > self.deadline:
            raise SearchLimit('Per-query time budget reached')


def walk_terms(term):
    yield term
    for child in term.args:
        yield from walk_terms(child)


def ground_clauses(clauses, budget):
    """Exact finite Herbrand grounding when there are no function symbols."""
    terms = [t for c in clauses for lit in c for a in lit.args for t in walk_terms(a)]
    if any(t.args for t in terms):
        raise InputError('Function terms require the resolution engine')
    constants = sorted({t for t in terms if not t.variable}) or [Term('__domain')]
    result = set()
    for clause in clauses:
        variables = sorted(clause_variables(clause))
        if len(constants) ** len(variables) + len(result) > budget.max_ground_clauses:
            raise SearchLimit('Grounding exceeded clause budget')
        for values in product(constants, repeat=len(variables)):
            budget.check()
            substitution = dict(zip(variables, values))
            ground = frozenset(substitute_literal(lit, substitution) for lit in clause)
            if not tautology(ground):
                result.add(ground)
    if any(lit.predicate == '=' for c in result for lit in c):
        add_equality_axioms(result, constants, budget)
    return sorted(result, key=lambda c: (len(c), tuple(sorted(c))))


def add_equality_axioms(clauses, constants, budget):
    """Equality is an equivalence relation respecting all ground predicates.

    Constants are NOT assumed different. A quotient model can identify them.
    Function terms with equality are rejected before this finite procedure.
    """
    def eq(a, b):
        return Literal('=', (a, b))
    def add(literals):
        budget.check()
        clauses.add(frozenset(literals))
        if len(clauses) > budget.max_ground_clauses:
            raise SearchLimit('Equality axioms exceeded clause budget')
    for a in constants:
        add([eq(a, a)])
        for b in constants:
            add([eq(a, b).opposite(), eq(b, a)])
            for c in constants:
                add([eq(a, b).opposite(), eq(b, c).opposite(), eq(a, c)])
    groups = defaultdict(set)
    for clause in list(clauses):
        for literal in clause:
            if literal.predicate != '=':
                groups[(literal.predicate, len(literal.args))].add(Literal(literal.predicate, literal.args))
    for atoms in groups.values():
        for a in atoms:
            for b in atoms:
                if a != b:
                    guards = [eq(x, y).opposite() for x, y in zip(a.args, b.args) if x != y]
                    add(guards + [a.opposite(), b])


## 3. Forward and backward chaining
Forward chaining starts with facts. Backward chaining starts with a goal.


In [3]:
from __future__ import annotations

def encode_clauses(clauses):
    atoms = sorted({Literal(l.predicate, l.args) for c in clauses for l in c})
    ids = {a: i + 1 for i, a in enumerate(atoms)}
    encoded = [frozenset((-1 if l.negative else 1) * ids[Literal(l.predicate, l.args)] for l in c) for c in clauses]
    return encoded, {i: str(a) for a, i in ids.items()}


#FORWARD CHAINING

def forward_chaining(clauses, names, budget):
    """Linear work-list forward chaining, including false-headed constraints."""
    # remaining[i] counts unmet premises; heads[i] is the conclusion.
    # dependents[p] lists the rules waiting for proposition p.
    remaining, heads, dependents = [], [], defaultdict(list)
    for i, clause in enumerate(clauses):
        body = [-literal for literal in clause if literal < 0]
        head = next((literal for literal in clause if literal > 0), None)
        remaining.append(len(body))
        heads.append(head)
        for premise in body:
            dependents[premise].append(i)
    ready = deque(i for i, n in enumerate(remaining) if n == 0)
    known, trace, steps = set(), [], 0
    while ready:
        budget.check()
        rule = ready.popleft()
        head = heads[rule]
        steps += 1
        if head is None:
            trace.append(f'C{rule + 1}: all premises hold; contradiction derived.')
            return False, {}, trace, steps
        if head in known:
            continue
        known.add(head)
        premises = [names[-l] for l in sorted(clauses[rule]) if l < 0]
        trace.append(f'C{rule + 1}: ' + (' & '.join(premises) + ' -> ' if premises else 'given ') + names[head])
        for dependent in dependents[head]:
            remaining[dependent] -= 1
            if remaining[dependent] == 0:
                ready.append(dependent)
    return True, {i: i in known for i in names}, trace, steps

#BACKWARD CHAINING


def backward_chaining(clauses, names, budget):
    """Prove Horn goals by working backwards through rules with matching heads.

    Only proved goals are cached. A failed cyclic branch is not a proof that
    the same goal must fail through a different rule.
    """
    rules_by_head = defaultdict(list)
    constraints = []
    for index, clause in enumerate(clauses):
        premises = tuple(sorted(-literal for literal in clause if literal < 0))
        head = next((literal for literal in clause if literal > 0), None)
        if head is None:
            constraints.append((index, premises))
        else:
            rules_by_head[head].append((index, premises))

    proved = set()
    trace = []
    steps = 0

    def prove_goal(goal, active_goals):
        nonlocal steps
        budget.check()
        steps += 1
        if goal in proved:
            return True
        if goal in active_goals:
            return False
        path = active_goals | {goal}
        for rule_id, premises in rules_by_head[goal]:
            if all(prove_goal(premise, path) for premise in premises):
                proved.add(goal)
                reason = ' & '.join(names[premise] for premise in premises) or 'given fact'
                trace.append(f'C{rule_id + 1}: proved {names[goal]} from {reason}')
                return True
        return False

    # A clause with no positive literal is a constraint. Proving all its
    # premises gives a contradiction, including the negated query constraint.
    for rule_id, premises in constraints:
        if all(prove_goal(premise, set()) for premise in premises):
            trace.append(f'C{rule_id + 1}: constraint contradicted.')
            return False, {}, trace, steps

    # A satisfying assignment is useful when the query is not entailed.
    model = {symbol: prove_goal(symbol, set()) for symbol in names}
    return True, model, trace, steps


## 4. DPLL and resolution
DPLL checks Boolean clauses. Resolution combines clauses with opposite literals.


In [4]:
from __future__ import annotations

#DPLL and resolution

def dpll(clauses, names, budget):
    """DPLL: unit propagation, pure literals, then deterministic branching."""
    trace, nodes = [], 0

    def note(message):
        if len(trace) < 60:
            trace.append(message)

    def simplify(current, literal):
        return [clause - {-literal} for clause in current if literal not in clause]

    def search(current, assignment, depth):
        nonlocal nodes
        budget.check()
        nodes += 1
        # First satisfy forced unit clauses, such as {A} or {~B}.
        while True:
            if not current:
                return assignment
            if any(not clause for clause in current):
                note(f'depth {depth}: empty clause; backtrack')
                return None
            unit = next((next(iter(c)) for c in current if len(c) == 1), None)
            if unit is None:
                break
            assignment[abs(unit)] = unit > 0
            note(f'depth {depth}: unit {"" if unit > 0 else "~"}{names[abs(unit)]}')
            current = simplify(current, unit)
            budget.check()
        # A pure literal appears with only one sign, so it can safely be set.
        signs = set().union(*current)
        pure = sorted(lit for lit in signs if -lit not in signs)
        if pure:
            for literal in pure:
                assignment[abs(literal)] = literal > 0
                current = simplify(current, literal)
            return search(current, assignment, depth)
        # Choose a frequent literal; its truth value is still checked both ways.
        scores = Counter(literal for clause in current for literal in clause)
        choice = max(sorted(scores), key=lambda lit: scores[lit])
        # If the first branch fails, try the opposite truth value.
        for literal in (choice, -choice):
            note(f'depth {depth}: branch {"" if literal > 0 else "~"}{names[abs(literal)]}')
            result = search(simplify(current, literal), {**assignment, abs(literal): literal > 0}, depth + 1)
            if result is not None:
                return result
        return None

    model = search(clauses, {}, 0)
    return model is not None, model or {}, trace, nodes


def unify(left, right, substitution=None):
    """Robinson-style term unification with the essential occurs check."""
    substitution = dict(substitution or {})
    pending = list(zip(left, right))
    if len(left) != len(right):
        return None
    while pending:
        a, b = pending.pop()
        a, b = substitute_term(a, substitution), substitute_term(b, substitution)
        if a == b:
            continue
        if a.variable or b.variable:
            if not a.variable:
                a, b = b, a
            if a in term_variables(b):
                return None
            substitution[a] = b
        elif a.name == b.name and len(a.args) == len(b.args):
            pending.extend(zip(a.args, b.args))
        else:
            return None
    return substitution


def canonical_clause(clause):
    mapping = {}
    def rename(term):
        if term.variable:
            if term not in mapping:
                mapping[term] = Term('__n' + str(len(mapping)), variable=True)
            return mapping[term]
        return Term(term.name, tuple(rename(a) for a in term.args))
    return frozenset(Literal(l.predicate, tuple(rename(t) for t in l.args), l.negative) for l in sorted(clause))


def resolution(clauses, budget):
    """Fair given-clause resolution with factoring and standardizing apart.

    Returns SAT only after finite saturation. Bounded interruption is UNKNOWN.
    Equality with function symbols is deliberately unsupported (no paramodulation).
    """
    if any(l.predicate == '=' for c in clauses for l in c):
        raise InputError('Equality with function symbols is not supported')
    records, seen, active, heap, age = [], {}, [], [], deque()
    serial = count()

    def add(clause, parents=()):
        clause = canonical_clause(clause)
        if tautology(clause) or clause in seen:
            return None
        if len(records) >= budget.max_resolution_clauses:
            raise SearchLimit('Resolution exceeded clause budget')
        index = len(records)
        seen[clause] = index
        records.append((clause, parents))
        heapq.heappush(heap, (len(clause), index))
        age.append(index)
        return index

    def proof(index):
        needed = set()
        def visit(i):
            if i in needed:
                return
            needed.add(i)
            for parent in records[i][1]:
                visit(parent)
        visit(index)
        return [f'C{i + 1}: ' + (' | '.join(map(str, sorted(records[i][0]))) or 'EMPTY') +
                (' [from ' + ', '.join('C' + str(p + 1) for p in records[i][1]) + ']' if records[i][1] else ' [input CNF]') for i in sorted(needed)]

    for clause in clauses:
        add(clause)
    processed, iteration = set(), 0
    while len(processed) < len(records):
        budget.check()
        iteration += 1
        if iteration % 5 == 0:
            while age and age[0] in processed:
                age.popleft()
            index = age.popleft()
        else:
            while heap and heap[0][1] in processed:
                heapq.heappop(heap)
            index = heapq.heappop(heap)[1]
        processed.add(index)
        clause = records[index][0]
        if not clause:
            return False, {}, proof(index), len(records)
        # Factoring is needed for completeness of first-order resolution.
        literals = sorted(clause)
        for i, a in enumerate(literals):
            for b in literals[i + 1:]:
                if (a.predicate, a.negative) == (b.predicate, b.negative):
                    substitution = unify(a.args, b.args)
                    if substitution is not None:
                        add(frozenset(substitute_literal(l, substitution) for l in clause - {b}), (index,))
        for other_index in active + [index]:
            budget.check()
            other = records[other_index][0]
            suffix = str(next(serial))
            rename = {v: Term('__apart' + suffix + '_' + v.name, variable=True) for v in clause_variables(other)}
            other = frozenset(substitute_literal(l, rename) for l in other)
            for a in sorted(clause):
                for b in sorted(other):
                    if a.predicate != b.predicate or a.negative == b.negative:
                        continue
                    substitution = unify(a.args, b.args)
                    if substitution is not None:
                        resolvent = frozenset(substitute_literal(l, substitution) for l in (clause - {a}) | (other - {b}))
                        new_index = add(resolvent, (index, other_index))
                        if new_index is not None and not resolvent:
                            return False, {}, proof(new_index), len(records)
        active.append(index)
    return True, {}, ['Resolution saturated without the empty clause.'], len(records)


## 5. Read input descriptions
Read simple English, formal formulas and Boolean equation descriptions.


In [5]:
from __future__ import annotations

# Read input descriptions

"""Input adapters. They translate syntax; they never consult gold answers."""

import re
import time



def symbol(text):
    text = re.sub(r'[^\w]+', '_', text.strip().lower()).strip('_')
    if not text:
        raise InputError('Empty name')
    return 'n_' + text if text[0].isdigit() else text


def english_atom(sentence, default_subject=None):
    """Small controlled-English grammar, not an unrestricted language model.

    Examples: Bob is kind; someone is young; the tiger likes the dog.
    All predicate and entity names come from the supplied text.
    """
    text = sentence.strip(' .').lower()
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    if not words:
        raise InputError('Empty English atom')
    if default_subject and len(words) == 1:
        subject, phrase = default_subject, 'is ' + text
    elif words[0] in ('they', 'it', 'them'):
        if not default_subject:
            raise InputError('Pronoun has no subject: ' + text)
        subject, phrase = default_subject, ' '.join(words[1:])
    elif words[0] in ('the', 'a', 'an'):
        if len(words) < 3:
            raise InputError('Incomplete English atom: ' + text)
        subject, phrase = Term(symbol(words[1])), ' '.join(words[2:])
    elif words[0] in ('someone', 'something', 'anyone', 'anything', 'everyone', 'everything'):
        subject, phrase = Term('x', variable=True), ' '.join(words[1:])
    elif default_subject and words[0] in ('is', 'are', 'was', 'not'):
        subject, phrase = default_subject, text
    else:
        if len(words) < 2:
            raise InputError('Expected subject and predicate: ' + text)
        subject, phrase = Term(symbol(words[0])), ' '.join(words[1:])
    negative = False
    if phrase.startswith(('is ', 'are ', 'was ')):
        phrase = phrase.split(' ', 1)[1]
    for prefix in ('does not ', 'do not ', 'not '):
        if phrase.startswith(prefix):
            phrase = phrase[len(prefix):]
            negative = True
            break
    phrase = re.sub(r'^(a|an) ', '', phrase)
    if not phrase:
        raise InputError('Missing predicate')
    # Relations with a named/determined object get two arguments.
    relation = re.fullmatch(r'(\w+) (?:the|a|an) (\w+)', phrase)
    if relation:
        predicate, target = relation.groups()
        formula = atom(symbol(predicate), subject, Term(symbol(target)))
    else:
        formula = atom(symbol(phrase), subject)
    return (('not', formula) if negative else formula), subject


def english_expression(text, default_subject=None):
    # Explicit full atoms or a shared subject: "Bob is smart and kind".
    parts = re.split(r'\s+(and|or)\s+', text.strip(' .'), flags=re.I)
    first, subject = english_atom(parts[0], default_subject)
    result = first
    for i in range(1, len(parts), 2):
        other, _ = english_atom(parts[i + 1], subject)
        result = (parts[i].lower(), result, other)
    return result, subject


RULE_PATTERNS = (
    r'^if (.+?),? then (.+)$',
    r'^if (.+?), (.+)$',
    r'^when (.+?), (?:we can conclude )?(.+)$',
    r'^given that (.+?), it follows that (.+)$',
    r'^based on (.+?), it is evident that (.+)$',
    r'^assuming (.+?), we can say (.+)$',
)


def english_formula(sentence):
    sentence = sentence.strip(' .')
    for pattern in RULE_PATTERNS:
        match = re.match(pattern, sentence, re.I)
        if match:
            left, subject = english_expression(match.group(1))
            right, _ = english_expression(match.group(2), subject)
            result = ('implies', left, right)
            if subject.variable:
                result = ('all', subject, result)
            return result
    match = re.fullmatch(r'(all|no) (\w+)(?: things| people)? are (.+)', sentence, re.I)
    if match:
        quantifier, category, description = match.groups()
        x = Term('x', variable=True)
        right, _ = english_expression('is ' + description, x)
        if quantifier.lower() == 'no':
            right = ('not', right)
        return ('all', x, ('implies', atom(symbol(category), x), right))
    if sentence.lower().startswith(('if ', 'all ', 'no ', 'some ', 'every ', 'either ')):
        raise InputError('Unsupported English construction; supply formal logic: ' + sentence)
    formula, subject = english_expression(sentence)
    return ('all', subject, formula) if subject.variable else formula


def split_premises(value):
    if isinstance(value, list):
        if not all(isinstance(s, str) for s in value):
            raise InputError('Premises must be strings')
        return value
    if isinstance(value, str):
        return [line.strip() for line in value.splitlines() if line.strip() and not line.lstrip().startswith('#')]
    raise InputError('Premises must be a list or newline-separated string')


def parse_english_context(context):
    text = re.sub(r'\b(?:Facts|Rules):', '', context, flags=re.I)
    sentences = [s.strip() for s in re.split(r'[.\n]+', text) if s.strip()]
    return [english_formula(sentence) for sentence in sentences]


def looks_formal(text):
    return bool(re.search(r'[∀∃¬∧∨→↔⊕~&|]|\b(?:forall|exists)\b|->|\w+\s*\(', text))


# A Boolean causal expression is ('var', name), ('not', expr), or ('and/or', a, b).
def boolean_expression(text):
    text = re.sub(r'\btogether\b', '', text, flags=re.I).strip()
    # Reuse the formal Boolean parser, then enforce a propositional vocabulary.
    tree = parse_formula(text)
    def convert(node):
        if node[0] == 'atom':
            lit = node[1]
            if lit.args:
                raise InputError('Causal variables must be propositional')
            return ('var', lit.predicate)
        if node[0] in ('not', 'and', 'or'):
            return (node[0], *(convert(child) for child in node[1:]))
        raise InputError('Causal equations accept only not/and/or')
    return convert(tree)


def expression_variables(expression):
    if expression[0] == 'var':
        return {expression[1]}
    return set().union(*(expression_variables(e) for e in expression[1:]))


def assignments(text):
    result = {}
    for item in re.split(r',\s*(?:and\s+)?|\s+and\s+', text.strip(' .'), flags=re.I):
        item = re.sub(r'^(\w+)\s+(?:does\s+)?not\s+occurs?$', r'not \1', item.strip(), flags=re.I)
        item = re.sub(r'\s+occurs?$', '', item, flags=re.I)
        match = re.fullmatch(r'(not\s+)?([\w]+)', item.strip(), re.I)
        if not match:
            raise InputError('Expected variable or not variable: ' + item)
        negative, name = match.groups()
        value = not bool(negative)
        if name in result and result[name] != value:
            raise InputError('Conflicting assignments for ' + name)
        result[name] = value
    return result


def parse_counterbench(context, question):
    """Extract a deterministic Boolean SCM from an explicit causal description.

    'A causes B' means B := A; 'A causes not B' means B := not A.
    This is stronger than material implication and is used only in causal mode.
    """
    text = re.sub(r'^We know that\s+', '', context.strip(), flags=re.I)
    observations = {}
    observation = re.search(r'\bWe observed\s+(.+)$', text, re.I)
    if observation:
        observations = assignments(observation.group(1))
        text = text[:observation.start()]
    priors = {}
    def extract_prior(match):
        name, probability = match.groups()
        probability = float(probability)
        if not 0 <= probability <= 1:
            raise InputError('Bernoulli probability outside [0,1]')
        priors[name] = probability
        return ''
    text = re.sub(r'(\w+)\s*~\s*Bern\(([\d.]+)\)\.?', extract_prior, text, flags=re.I).strip(' .')
    # A comma starts a new equation only if the next clause includes cause(s).
    chunks = re.split(r',\s*(?:and\s+)?(?=[^,.]*\bcauses?\b)', text, flags=re.I)
    equations = {}
    for chunk in chunks:
        match = re.fullmatch(r'(.+?)\s+causes?\s+(.+)', chunk.strip(' .'), re.I)
        if not match:
            raise InputError('Cannot parse causal equation: ' + chunk)
        antecedent = boolean_expression(match.group(1))
        for target, value in assignments(match.group(2)).items():
            equation = antecedent if value else ('not', antecedent)
            if target in equations:
                raise InputError('Multiple equations for ' + target + '; provide one combined equation')
            equations[target] = equation
    question = question.strip()
    interventions, factual = {}, {}
    normal = re.fullmatch(r'Would (\w+) occur if (.+?)\?', question, re.I)
    nested = re.fullmatch(r'Assume (.+?), and based on this assumption, further suppose (.+?)\. Would (\w+) occur\?', question, re.I)
    if normal:
        target, intervention = normal.groups()
        chunks = re.split(r'\s+instead of\s+', intervention, flags=re.I)
        interventions = assignments(chunks[0])
        if len(chunks) == 2:
            factual = assignments(chunks[1])
    elif nested:
        first, second, target = nested.groups()
        interventions = {**assignments(first), **assignments(second)}
    else:
        raise InputError('Unsupported causal question')
    for name, value in factual.items():
        if name in observations and observations[name] != value:
            raise InputError('Factual condition contradicts observation')
        observations[name] = value
    return equations, target, interventions, observations, priors


def counterbench_formulas(equations, query, intervention, observations, priors):
    """Translate an explicit 'what if' problem into ordinary Boolean formulas.

    F__A means A in the factual situation; H__A means A after the change.
    Root__A connects an unchanged background variable in the two situations.
    The returned formulas are passed to DPLL.
    """
    if not isinstance(equations, dict) or not isinstance(query, str):
        raise InputError('Expected an equation dictionary and a query variable')
    equations = {
        name: boolean_expression(value) if isinstance(value, str) else value
        for name, value in equations.items()
    }
    variables = set(equations)
    for expression in equations.values():
        variables.update(expression_variables(expression))
    used_names = set(observations) | set(intervention) | set(priors) | {query}
    if used_names - variables:
        raise InputError('Unknown variable: ' + sorted(used_names - variables)[0])
    if not all(isinstance(v, bool) for v in [*observations.values(), *intervention.values()]):
        raise InputError('Observations and interventions require Boolean values')
    if not all(isinstance(p, (int, float)) and 0 <= p <= 1 for p in priors.values()):
        raise InputError('Prior values must be in [0,1]')
    roots = variables - set(equations)
    if set(priors) - roots:
        raise InputError('Priors may be attached only to background root variables')

    # Input validation: reject circular definitions. This does not infer facts.
    available = set(roots)
    pending = set(equations)
    while pending:
        ready = {name for name in pending if expression_variables(equations[name]) <= available}
        if not ready:
            raise InputError('Cyclic definitions are outside the supported input format')
        available.update(ready)
        pending.difference_update(ready)

    def in_world(expression, prefix):
        if expression[0] == 'var':
            return atom(prefix + expression[1])
        return (expression[0], *(in_world(child, prefix) for child in expression[1:]))

    def assigned(name, value):
        return atom(name) if value else ('not', atom(name))

    formulas = []
    for name in sorted(variables):
        factual = atom('F__' + name)
        hypothetical = atom('H__' + name)
        if name in equations:
            # A structural equation is represented by a Boolean equivalence.
            formulas.append(('iff', factual, in_world(equations[name], 'F__')))
        else:
            formulas.append(('iff', factual, atom('Root__' + name)))

        if name in intervention:
            # Replace this equation in the hypothetical situation only.
            formulas.append(assigned('H__' + name, intervention[name]))
        elif name in equations:
            formulas.append(('iff', hypothetical, in_world(equations[name], 'H__')))
        else:
            formulas.append(('iff', hypothetical, atom('Root__' + name)))

    for name, value in sorted(observations.items()):
        formulas.append(assigned('F__' + name, value))
    for name, probability in sorted(priors.items()):
        # Only impossible/certain root values affect universal entailment.
        if probability in (0, 1):
            formulas.append(assigned('Root__' + name, probability == 1))
    return formulas, atom('H__' + query)


## 6. Prediction pipeline
Choose the method from the clauses and return the answer, status and reasoning steps.


In [6]:
from __future__ import annotations

# 6. Prediction pipeline

def validate_signature(formulas):
    arities = {}
    def visit(formula):
        if formula[0] == 'atom':
            literal = formula[1]
            if literal.predicate in arities and arities[literal.predicate] != len(literal.args):
                raise InputError('Inconsistent predicate arity: ' + literal.predicate)
            arities[literal.predicate] = len(literal.args)
        elif formula[0] in ('all', 'exists'):
            visit(formula[2])
        else:
            for f in formula[1:]:
                visit(f)
    for f in formulas:
        visit(f)


def check_satisfiability(formulas, budget, use_dpll=False, horn_method='auto'):
    if horn_method not in ('auto', 'forward_chaining', 'backward_chaining'):
        raise InputError('Unknown Horn method: ' + str(horn_method))
    validate_signature(formulas)
    clauses = Clausifier().convert(formulas)
    budget.check()
    functions = any(t.args for c in clauses for l in c for a in l.args for t in walk_terms(a))
    if functions:
        sat, model, trace, steps = resolution(clauses, budget)
        return sat, model, trace, steps, 'resolution', len(clauses)
    ground = ground_clauses(clauses, budget)
    encoded, names = encode_clauses(ground)
    horn = all(sum(l > 0 for l in clause) <= 1 for clause in encoded)
    engine = 'DPLL'
    solver = dpll
    if horn and not use_dpll:
        # One negative clause gives a single goal for backward chaining.
        # Several constraints are handled together by forward chaining.
        constraints = sum(not any(literal > 0 for literal in clause) for clause in encoded)
        engine = horn_method
        if engine == 'auto':
            engine = 'backward_chaining' if constraints == 1 else 'forward_chaining'
        solver = backward_chaining if engine == 'backward_chaining' else forward_chaining
    sat, model, trace, steps = solver(encoded, names, budget)
    named_model = {names[i]: model.get(i, False) for i in names} if sat else {}
    return sat, named_model, trace, steps, engine, len(ground)


def prove(premises, query, seconds=2.0, check_consistency=False, horn_method='auto'):
    """Return binary entailment and an explicit diagnostic status.

    The Boolean is True exactly when KB & ~query is proved inconsistent.
    An input error or stopped search returns None, meaning unresolved.
    """
    start = time.perf_counter()
    result = {'prediction': None, 'status': 'input_error', 'engine': None,
              'trace': [], 'countermodel': {}, 'steps': 0, 'clauses': 0}
    try:
        budget = Budget(seconds)
        premises = [parse_formula(p) if isinstance(p, str) else p for p in premises]
        query = parse_formula(query) if isinstance(query, str) else query
        sat, model, trace, steps, engine, size = check_satisfiability(
            premises + [('not', query)], budget, horn_method=horn_method)
        result.update(prediction=not sat, status='entailed' if not sat else 'not_entailed',
                      engine=engine, trace=trace, countermodel=model, steps=steps, clauses=size)
        if check_consistency and not sat:
            try:
                kb_sat, *_ = check_satisfiability(premises, budget, horn_method=horn_method)
                result['kb_consistent'] = kb_sat
                if not kb_sat:
                    result['status'] = 'inconsistent_kb'
                    result['trace'].append('The KB itself is inconsistent; classical explosion applies.')
            except (SearchLimit, RecursionError):
                result['kb_consistent'] = None
    except (SearchLimit, RecursionError) as error:
        result.update(status='resource_limit', error=str(error))
    except InputError as error:
        result.update(status='input_error', error=str(error))
    result['seconds'] = time.perf_counter() - start
    return result

def solve_counterbench(equations, query, intervention, observations=None, priors=None, seconds=2.0):
    """Use DPLL on the translated formulas; no separate causal search engine."""
    start = time.perf_counter()
    result = {'prediction': None, 'status': 'input_error', 'engine': 'DPLL',
              'trace': [], 'countermodel': {}, 'steps': 0, 'clauses': 0}
    try:
        budget = Budget(seconds)
        formulas, goal = counterbench_formulas(
            equations, query, intervention, observations or {}, priors or {})

        # First check that the factual evidence is possible. This prevents a
        # contradictory input from being mistaken for an ordinary positive case.
        possible, _, _, first_steps, _, _ = check_satisfiability(formulas, budget, use_dpll=True)
        if not possible:
            result.update(status='inconsistent_evidence',
                          error='No assignment satisfies the factual observations')
        else:
            # The same entailment test as other Boolean problems: KB AND NOT Q.
            satisfiable, model, trace, steps, engine, size = check_satisfiability(
                formulas + [('not', goal)], budget, use_dpll=True)
            result.update(prediction=not satisfiable,
                          status='not_entailed' if satisfiable else 'entailed',
                          countermodel=model, steps=first_steps + steps, clauses=size)
            result['trace'] = ['F__ names describe the factual situation; H__ names describe the changed situation.',
                               'DPLL checked the equations and the negated query.'] + trace
    except (SearchLimit, RecursionError) as error:
        result.update(status='resource_limit', error=str(error))
    except InputError as error:
        result.update(status='input_error', error=str(error))
    result['seconds'] = time.perf_counter() - start
    return result


def predict(context, statement, input_format='auto', seconds=2.0,
            check_consistency=False, horn_method='auto'):
    """Simple evaluator interface: predict(context, statement)['prediction']."""
    start = time.perf_counter()
    try:
        if not isinstance(statement, str) or not statement.strip():
            raise InputError('A nonempty query string is required')
        if not isinstance(context, (str, list)) or (isinstance(context, list) and not all(isinstance(x, str) for x in context)):
            raise InputError('Context must be a string or list of formula strings')
        if input_format not in ('auto', 'logic', 'english'):
            raise InputError('input_format must be auto, logic, or english')
        if input_format == 'auto':
            combined = '\n'.join(context) if isinstance(context, list) else context
            input_format = 'logic' if looks_formal(combined + '\n' + statement) else 'english'
        if input_format == 'logic':
            premises = split_premises(context)
            result = prove(premises, statement, seconds=seconds,
                           check_consistency=check_consistency, horn_method=horn_method)
        else:
            if isinstance(context, list):
                context = '\n'.join(context)
            result = prove(parse_english_context(context), english_formula(statement),
                           seconds=seconds, check_consistency=check_consistency, horn_method=horn_method)
    except InputError as error:
        result = {'prediction': None, 'status': 'input_error', 'engine': None, 'error': str(error),
                  'trace': [], 'countermodel': {}, 'steps': 0, 'clauses': 0}
    result['seconds'] = time.perf_counter() - start
    result['input_format'] = input_format
    return result


def read_record(record, seconds=2.0):
    """Only input fields are accessed. Labels/reasoning/ids are never read."""
    if 'equations' in record:
        return solve_counterbench(record['equations'], record['query'], record.get('intervention', {}),
                            record.get('observations', {}), record.get('priors', {}), seconds)
    if 'given_info' in record and 'question' in record:
        start = time.perf_counter()
        try:
            parsed = parse_counterbench(record['given_info'], record['question'])
            result = solve_counterbench(*parsed, seconds=seconds)
        except InputError as error:
            result = {'prediction': None, 'status': 'input_error', 'engine': 'DPLL',
                      'error': str(error), 'trace': [], 'countermodel': {}, 'steps': 0, 'clauses': 0}
        result['seconds'] = time.perf_counter() - start
        return result
    if 'premises-FOL' in record and 'conclusion-FOL' in record:
        return predict(record['premises-FOL'], record['conclusion-FOL'], 'logic', seconds)
    if 'facts' in record or 'rules' in record:
        premises = split_premises(record.get('facts', [])) + split_premises(record.get('rules', []))
        return predict(premises, record.get('query', record.get('statement', '')), 'logic', seconds)
    context = record.get('premises', record.get('context', record.get('kb', [])))
    statement = record.get('query', record.get('statement', record.get('conclusion', record.get('question', ''))))
    return predict(context, statement, record.get('input_format', 'auto'), seconds)


def predict_record(record, seconds=2.0):
    """Read one record; report missing or invalid fields as an input error."""
    start = time.perf_counter()
    try:
        if not isinstance(record, dict):
            raise InputError('Each input record must be a dictionary')
        return read_record(record, seconds)
    except (InputError, KeyError, TypeError) as error:
        return {'prediction': None, 'status': 'input_error', 'engine': None,
                'error': str(error), 'trace': [], 'countermodel': {},
                'steps': 0, 'clauses': 0, 'seconds': time.perf_counter() - start}


def predict_flag(context, statement, input_format='auto', seconds=2.0):
    """Return a proved Boolean result; raise an error if the case is unresolved."""
    result = predict(context, statement, input_format, seconds)
    if result['prediction'] is None:
        raise ValueError(result['status'] + ': ' + result.get('error', 'Unresolved query'))
    return result['prediction']


## 7. Examples and checks
The examples show all four methods. Incorrect input is reported as unresolved.


In [7]:
from __future__ import annotations

# Examples and checks

def show_examples():
    facts = ['Young(bob)', 'Kind(bob)', 'forall x (Young(x) -> Smart(x))',
             'forall x ((Smart(x) & Kind(x)) -> Successful(x))']
    examples = {
        'Forward chaining': prove(facts, 'Successful(bob)', horn_method='forward_chaining'),
        'Backward chaining': prove(facts, 'Successful(bob)', horn_method='backward_chaining'),
        'DPLL': prove(['A | B'], 'A'),
        'Resolution': prove(['P(a)', 'forall x (P(x) -> P(f(x)))'], 'P(f(f(a)))'),
    }
    for name, result in examples.items():
        print('\n' + name + ': ' + str(result['prediction']))
        for step in result['trace']:
            print(' ', step)
        if result['countermodel']:
            print(' Counterexample:', result['countermodel'])
    return examples


def run_basic_checks():
    """Small checks that can be rerun with the notebook."""
    cases = [
        (['A', 'A -> B'], 'B', True),
        (['B', 'A -> B'], 'A', False),
        (['A -> B', 'B -> A'], 'A', False),
        (['B -> A', 'A -> B', 'C -> A', 'C'], 'B', True),
        (['P(iris)'], '~Q(iris)', False),
        (['A', '~B', '~C'], 'A', True),
    ]
    for premises, query, expected in cases:
        for method in ('forward_chaining', 'backward_chaining'):
            result = prove(premises, query, horn_method=method)
            assert result['prediction'] == expected, result
    assert prove(['A | B'], 'A')['prediction'] is False
    assert prove(['P(a)', 'forall x (P(x) -> P(f(x)))'], 'P(f(f(a)))')['prediction'] is True
    assert predict(['P(a))'], 'P(a)', 'logic')['status'] == 'input_error'
    row = {'facts': ['A', 'A -> B'], 'query': 'B', 'label': False, 'id': 'arbitrary'}
    assert predict_record(row)['prediction'] is True
    print('Passed 16 basic checks.')


In [8]:
run_basic_checks()
worked_examples = show_examples()


Passed 16 basic checks.

Forward chaining: True
  C1: given Kind(bob)
  C3: given Young(bob)
  C4: Young(bob) -> Smart(bob)
  C5: Smart(bob) & Kind(bob) -> Successful(bob)
  C2: all premises hold; contradiction derived.

Backward chaining: True
  C1: proved Kind(bob) from given fact
  C3: proved Young(bob) from given fact
  C4: proved Smart(bob) from Young(bob)
  C5: proved Successful(bob) from Kind(bob) & Smart(bob)
  C2: constraint contradicted.

DPLL: False
  depth 0: unit ~A
  depth 0: unit B
 Counterexample: {'A': False, 'B': True}

Resolution: True
  C1: P(a) [input CNF]
  C2: ~P(?__n0) | P(f(?__n0)) [input CNF]
  C3: ~P(f(f(a))) [input CNF]
  C4: P(f(a)) [from C2, C1]
  C5: ~P(f(a)) [from C2, C3]
  C9: EMPTY [from C5, C4]


## 8. Evaluate datasets and write CSV
Labels are used only for scoring. The combined CSV contains every record from the selected files.


In [9]:
from __future__ import annotations

# Evaluate datasets and write CSV

"""Reproducible public evaluation and label-blind prediction-file generation."""
from collections import Counter
import csv
import hashlib
import json
from pathlib import Path
import platform
import statistics
import time



def load_rows(path):
    path = Path(path)
    with path.open(encoding='utf-8-sig', newline='') as stream:
        if path.suffix.lower() == '.csv':
            rows = list(csv.DictReader(stream))
            for row in rows:
                for key in ('premises', 'facts', 'rules', 'equations', 'intervention', 'observations', 'priors'):
                    if row.get(key, '').lstrip().startswith(('[', '{')):
                        row[key] = json.loads(row[key])
            return rows
        if path.suffix.lower() == '.jsonl':
            return [json.loads(line) for line in stream if line.strip()]
        data = json.load(stream)
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and 'data' in data:
        return data['data']
    if isinstance(data, dict):
        return [data]
    raise ValueError('Expected JSON record, list, data wrapper, JSONL, or CSV')


def gold_label(row):
    value = next((row[k] for k in ('label', 'flag', 'answer') if k in row), None)
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    value = str(value).strip().lower()
    if value in ('true', 'yes', 'entailment', 'entailed', '1'):
        return True
    if value in ('false', 'no', 'uncertain', 'unknown', 'not entailment', 'neutral', 'contradiction', '0'):
        return False
    raise ValueError('Unrecognized gold label: ' + value)


def record_id(row, index):
    return str(next((row[k] for k in ('id', 'example_id', 'example-id', 'question_id') if k in row), index))


INPUT_FIELDS = {'equations', 'query', 'intervention', 'observations', 'priors', 'given_info', 'question',
                'premises-FOL', 'conclusion-FOL', 'facts', 'rules', 'premises', 'context', 'kb',
                'statement', 'conclusion', 'input_format'}


def input_only(row):
    return {key: value for key, value in row.items() if key in INPUT_FIELDS}


def evaluate_rows(rows, name='dataset', seconds=2.0):
    start = time.perf_counter()
    diagnostics = []
    for index, row in enumerate(rows):
        # Strip gold labels, gold reasoning traces, metadata, and identifiers.
        result = predict_record(input_only(row), seconds)
        result['id'] = record_id(row, index)
        result['gold'] = gold_label(row)
        result['used_fallback'] = result['prediction'] is None
        result['output_label'] = csv_label(result)
        diagnostics.append(result)
    elapsed = time.perf_counter() - start
    labeled = [r for r in diagnostics if r['gold'] is not None]
    correct = sum(r['output_label'] == r['gold'] for r in labeled)
    resolved = {'entailed', 'not_entailed', 'inconsistent_kb'}
    times = sorted(r['seconds'] for r in diagnostics)
    metrics = {'dataset': name, 'rows': len(rows), 'labeled': len(labeled), 'correct': correct,
               'accuracy': correct / len(labeled) if labeled else None,
               'resolved': sum(r['status'] in resolved for r in diagnostics),
               'coverage': sum(r['status'] in resolved for r in diagnostics) / len(rows) if rows else 0,
               'prediction_seconds': sum(times), 'evaluation_seconds': elapsed,
               'median_ms': statistics.median(times) * 1000 if times else 0,
               'p95_ms': times[min(len(times) - 1, int(len(times) * .95))] * 1000 if times else 0,
               'statuses': dict(Counter(r['status'] for r in diagnostics)),
               'engines': dict(Counter(str(r['engine']) for r in diagnostics)),
               'confusion': dict(Counter(('T' if r['gold'] else 'F') + ('T' if r['output_label'] else 'F') for r in labeled)),
               'seconds_budget': seconds}
    answered = [r for r in labeled if r['prediction'] is not None]
    metrics['answered_correct'] = sum(r['prediction'] == r['gold'] for r in answered)
    metrics['answered_accuracy'] = metrics['answered_correct'] / len(answered) if answered else None
    metrics['unresolved'] = sum(r['used_fallback'] for r in diagnostics)
    metrics['fallback_correct'] = sum(r['used_fallback'] and r['output_label'] == r['gold'] for r in labeled)
    return metrics, diagnostics


def csv_label(result):
    """The required CSV is Boolean; keep unresolved status in the result."""
    return False if result['prediction'] is None else result['prediction']


def write_predictions(rows, path, seconds=2.0, label_column='predicted_label', diagnostics_path=None):
    """Unlabeled evaluation is supported. IDs are preserved; duplicates fail."""
    ids = [record_id(row, i) for i, row in enumerate(rows)]
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate IDs: use one evaluation file at a time or namespace IDs')
    records = []
    for identifier, row in zip(ids, rows):
        result = predict_record(input_only(row), seconds)
        records.append({'id': identifier, **result, 'output_label': csv_label(result),
                        'used_fallback': result['prediction'] is None})
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['id', label_column])
        writer.writerows((r['id'], str(r['output_label'])) for r in records)
    if diagnostics_path:
        with Path(diagnostics_path).open('w', encoding='utf-8') as stream:
            for record in records:
                stream.write(json.dumps(record, ensure_ascii=False) + '\n')
    return records

"""Evaluate input files and write one combined prediction file."""


def run_datasets(dataset_files, data_dir, output_path, seconds=1.0):
    data_dir = Path(data_dir)
    summary, details, combined = [], {}, []
    multiple_files = len(dataset_files) > 1

    print(f'{"Dataset":32} {"Rows":>6} {"Correct":>8} {"Accuracy":>10} {"Unresolved":>11} {"Time(s)":>9}')
    for filename in dataset_files:
        path = data_dir / filename
        rows = load_rows(path)
        measured, results = evaluate_rows(rows, filename, seconds)
        measured['sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
        summary.append(measured)
        details[filename] = results
        accuracy = f'{measured["accuracy"]:.2%}' if measured['accuracy'] is not None else 'N/A'
        print(f'{filename:32} {len(rows):6} {measured["correct"]:8} {accuracy:>10} '
              f'{measured["unresolved"]:11} {measured["prediction_seconds"]:9.3f}', flush=True)

        for result in results:
            original_id = result['id']
            # File prefixes keep IDs unique when several public splits are combined.
            identifier = f'{Path(filename).stem}:{original_id}' if multiple_files else original_id
            combined.append({**result, 'id': identifier, 'source_id': original_id, 'dataset': filename})

    ids = [result['id'] for result in combined]
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate IDs found; CSV was not written')
    with Path(output_path).open('w', encoding='utf-8', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['id', 'predicted_label'])
        writer.writerows((row['id'], str(row['output_label'])) for row in combined)

    total_rows = sum(row['rows'] for row in summary)
    total_labeled = sum(row['labeled'] for row in summary)
    total_correct = sum(row['correct'] for row in summary)
    total_seconds = sum(row['prediction_seconds'] for row in summary)
    overall_accuracy = total_correct / total_labeled if total_labeled else None
    display_accuracy = f'{overall_accuracy:.2%}' if overall_accuracy is not None else 'N/A'
    print(f'Total: {total_rows} records | Accuracy: {display_accuracy} | Time: {total_seconds:.3f}s')
    print('Saved:', Path(output_path).resolve())
    print('Methods:', dict(Counter(row['engine'] for row in combined)))
    return summary, details, combined


In [10]:
# Change BASE_DIR if the notebook kernel starts outside this folder.
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Data"
OUTPUT_FILE = BASE_DIR / "submission.csv"
SECONDS_PER_QUERY = 1.0

dataset_files = [
    "ruletaker_train.jsonl",
    "ruletaker_test.jsonl",
    "ruletaker_ood_test.jsonl",
    "counterbench_alpha.json",
    "counterbench_backdoor.json",
    "FOLIO_train.jsonl",
    "FOLIO_validation.jsonl",
]

summary, details, predictions = run_datasets(
    dataset_files, DATA_DIR, OUTPUT_FILE, SECONDS_PER_QUERY)

Dataset                            Rows  Correct   Accuracy  Unresolved   Time(s)
ruletaker_train.jsonl              1000      983     98.30%           0     2.005
ruletaker_test.jsonl                250      244     97.60%           0     0.577
ruletaker_ood_test.jsonl            375      373     99.47%           0     0.737
counterbench_alpha.json            1000      905     90.50%         165     4.839
counterbench_backdoor.json          200      175     87.50%          10     1.340
FOLIO_train.jsonl                  1001      774     77.32%         236    65.836
FOLIO_validation.jsonl              203      164     80.79%          62     6.089
Total: 4029 records | Accuracy: 89.80% | Time: 81.423s
Saved: C:\Users\amank\Downloads\AI_FINAL_M26MAC001\submission\submission.csv
Methods: {'backward_chaining': 1750, 'DPLL': 1611, None: 298, 'forward_chaining': 344, 'resolution': 26}
